In [38]:
%matplotlib inline

In [39]:
import pandas as pd
import numpy as np
import json

In [40]:
df = pd.read_csv("movies.csv")
missing_values = df.isna().sum()
missing_values

,0
index,0
budget,0
genres,28
homepage,3091
id,0
keywords,412
original_language,0
original_title,0
overview,3
popularity,0


In [41]:
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

In [42]:
movie_title_map = {}
movie_genre_map = {}
movie_popularity_map = {}

df_raw = pd.read_csv("movies.csv")
df_raw = df_raw.dropna(subset=["genres", "popularity"])

for _, row in df_raw.iterrows():
    movie_id = row["title"]
    movie_title_map[movie_id] = row["title"]

    this_movie_genres = {}
    for g in row["genres"].split("|"):
        g = g.strip()
        if g:
            this_movie_genres[g] = this_movie_genres.get(g, 0) + 1

    movie_genre_map[movie_id] = this_movie_genres
    movie_popularity_map[movie_id] = row["popularity"]

In [43]:
index = movie_genre_map.keys()
rows = [movie_genre_map[k] for k in index]
df = pd.DataFrame(rows, index=index)
df = df.fillna(0)
df

,Action Adventure Fantasy Science Fiction,Adventure Fantasy Action,Action Adventure Crime,Action Crime Drama Thriller,Action Adventure Science Fiction,Fantasy Action Adventure,Animation Family,Adventure Fantasy Family,Action Adventure Fantasy,Adventure Fantasy Action Science Fiction,...,Documentary History,Science Fiction Drama Mystery,Drama Fantasy Horror Science Fiction,Documentary Comedy Drama,Mystery Horror Thriller,Horror Comedy Crime,Crime Horror Mystery Thriller,Thriller Horror Comedy,Foreign Thriller,Comedy Drama Romance TV Movie
Avatar,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Pirates of the Caribbean: At World's End,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Spectre,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
The Dark Knight Rises,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
John Carter,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cavite,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
El Mariachi,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Newlyweds,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"Signed, Sealed, Delivered",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [44]:
from sklearn.preprocessing import MinMaxScaler
import scipy.spatial.distance

scaler = MinMaxScaler()
popularity_values = np.array([movie_popularity_map[k] for k in index]).reshape(-1, 1)
popularity_scaled = scaler.fit_transform(popularity_values)

features_df = pd.DataFrame(
    np.hstack([df.values, popularity_scaled]),
    index=index
)

In [45]:
target_movie_id = "Armageddon"

In [46]:
target_movie_features = features_df.loc[target_movie_id]

distances = scipy.spatial.distance.cdist(features_df, [target_movie_features], metric="cosine")[:,0]

query_distances = list(zip(features_df.index, distances))

In [47]:
print(f"Selected Movie: {target_movie_id}")
print(f"Genre: {df_raw[df_raw['title'] == target_movie_id]['genres'].values[0]}")
print(f"Popularity Score: {movie_popularity_map[target_movie_id]:.2f}\n")

print("Top Recommendations:")
for similar_movie_id, similar_score in sorted(query_distances, key=lambda x: x[1], reverse=False)[1:11]:
    similar_genres = movie_genre_map[similar_movie_id]
    similar_popularity = movie_popularity_map[similar_movie_id]
    print(similar_movie_id, f"| Genre: {similar_genres} | Popularity: {similar_popularity:.2f}")

Selected Movie: Armageddon
Genre: Action Thriller Science Fiction Adventure
Popularity Score: 58.49

Top Recommendations:
I Am Number Four | Genre: {'Action Thriller Science Fiction Adventure': 1} | Popularity: 43.45
The Island | Genre: {'Action Thriller Science Fiction Adventure': 1} | Popularity: 37.68
Minions | Genre: {'Family Animation Adventure Comedy': 1} | Popularity: 875.58
Interstellar | Genre: {'Adventure Drama Science Fiction': 1} | Popularity: 724.25
Deadpool | Genre: {'Action Adventure Comedy': 1} | Popularity: 514.57
Guardians of the Galaxy | Genre: {'Action Science Fiction Adventure': 1} | Popularity: 481.10
Mad Max: Fury Road | Genre: {'Action Adventure Science Fiction Thriller': 1} | Popularity: 434.28
Jurassic World | Genre: {'Action Adventure Science Fiction Thriller': 1} | Popularity: 418.71
Pirates of the Caribbean: The Curse of the Black Pearl | Genre: {'Adventure Fantasy Action': 1} | Popularity: 271.97
Dawn of the Planet of the Apes | Genre: {'Science Fiction Ac